In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rollsroyce1/
      /kaggle/input/datasets/rollsroyce1/sat-ext/
        /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/rollsroyce1/sat-results-03/
        /kaggle/input/datasets/rollsroyce1/sat-results-03/results/
      /kaggle/input/datasets/rollsroyce1/sat-results-07a/
        yolo11s.pt
        yolo26n.pt
        /kaggle/input/datasets/rollsroyce1/sat-results-07a/results/
        /kaggle/input/datasets/rollsroyce1/sat-results-07a/results_07A/
        /kaggle/input/datasets/rollsroyce1/sat-results-07a/weights/
          grouped__YOLO11s-1024__s42_best.pt
          grouped__YOLO11s-1024__s43_best.pt
          grouped__YOLO11s-1024__s44_best.pt
          grouped__YOLO11s-1280__s42_best.pt
          grouped__YOLO11s-1280__s43_best.pt
       

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
utils.warn_if_gpu_wasted()
utils.pip_install("ensemble-boxes")
import report, analysis
runs = report.collect_runs()
df = report.long_table(runs)
os.makedirs(report.PAPER_DIR, exist_ok=True)

126 evaluated runs found (+52 TTA / inference-resolution modes from 08D)


In [4]:
err = analysis.table_errors(df, "grouped")
print(analysis.fig_errors(err, "grouped"))
if (df.split == "sky_grouped").any():
    analysis.fig_errors(analysis.table_errors(df, "sky_grouped"), "sky_grouped")


### T11_errors_grouped: Error breakdown at conf $\geq$ 0.25 (TIDE definitions, fg IoU 0.5, bg IoU 0.1), mean over seeds. %FP = share of all false positives.
    Model Inference  Seeds     TP  duplicate  classification  localization  cls+loc  background  missed  missed_micro  missed_tiny16_32  missed_medium  missed_large  duplicate_%FP  classification_%FP  localization_%FP  cls+loc_%FP  background_%FP
  YOLO11n  Standard      3 1585.0      120.7            14.3         195.0      3.0       338.3  1150.0         718.0             312.3          101.0          18.7           18.0                 2.1              29.0          0.4            50.4
  YOLO11n  SAHI-480      3 1811.0       39.7            24.3         456.0      9.7       566.7   924.0         605.7             217.3           84.0          17.0            3.6                 2.2              41.6          0.9            51.7
  YOLO11n  SAHI-320      3 1864.0       57.3            36.0         709.0     26.7      1244.7   871

In [5]:
analysis.table_mcnemar(df, "grouped", "standard")
analysis.table_mcnemar_modes(df, "grouped")
if "sky_grouped" in set(df.split):
    analysis.table_mcnemar(df, "sky_grouped", "standard"); analysis.table_mcnemar_modes(df, "sky_grouped")
# native resolution (07J-07L): native_tiles has the SAME val/test images and ids as native_grouped -> one paired table
nat = df[df.split.isin(["native_grouped", "native_tiles"])].copy()
if len(nat):
    nat["model"] = nat.model.astype(str) + np.where(nat.split == "native_tiles", " (tiles)", "")
    nat["split"] = "native_grouped"
    analysis.fig_errors(analysis.table_errors(nat, "native_grouped", models=sorted(set(nat.model))), "native_grouped")
    analysis.table_mcnemar(nat, "native_grouped", "standard", models=sorted(set(nat.model)))
    analysis.table_mcnemar_modes(nat, "native_grouped", models=sorted(set(nat.model)))
    analysis.table_ensembles(df, "native_grouped")     # WBF of seeds / architectures at native resolution
else:
    print("no native runs attached (07J-07L) -> native McNemar skipped")


### T12_mcnemar_grouped_standard: Pairwise comparison of models (grouped test split, Standard inference). McNemar: exact test on objects detected (conf $\geq$ 0.25, IoU $\geq$ 0.5) by the majority of seeds; objects in one image are correlated, so the image-level paired bootstrap of the recall difference is the primary test. Holm correction over all pairs.
           A            B  Objects  Only_A  Only_B     p_McNemar  Recall_B_minus_A CI95_image_bootstrap  p_bootstrap  p_McNemar_Holm  p_bootstrap_Holm
     YOLO11n      YOLO11s     2735      74     278  6.370000e-29            0.0585   [+0.0336, +0.0846]        0.000    8.660000e-27             0.000
     YOLO11n      YOLO11m     2735     113     310  2.650000e-22            0.0641   [+0.0198, +0.1127]        0.001    3.310000e-20             0.104
     YOLO11n      YOLOv8m     2735      99     308  4.730000e-26            0.0709   [+0.0292, +0.1142]        0.000    6.150000e-24             0.000
     YOLO11n    RT-DETR-L     2735   

In [6]:
analysis.table_ensembles(df, "grouped")
import shutil
print(shutil.make_archive(os.path.join(cfg.WORK_DIR, "paper_analysis"), "zip", report.PAPER_DIR))


### T13_ensembles_grouped: Weighted Box Fusion (Solovyev et al., 2021; IoU 0.55) of seeds, of architectures, and of standard + SAHI-320 predictions (grouped test split).
                         Ensemble                                   Members     AP   AP50  AP_small  AR50_micro          TP          FP      R      P
   YOLO11n single (mean of seeds)                                         1 0.3614 0.7079    0.2780      0.4689 1585.000000  671.333333 0.5795 0.7024
           YOLO11n WBF of 3 seeds                                   3 seeds 0.3673 0.7312    0.2848      0.4875 1341.000000  255.000000 0.4903 0.8402
   YOLO11s single (mean of seeds)                                         1 0.3799 0.7427    0.3084      0.4942 1745.000000  613.000000 0.6380 0.7420
           YOLO11s WBF of 3 seeds                                   3 seeds 0.3908 0.7696    0.3257      0.5379 1516.000000  248.000000 0.5543 0.8594
   YOLO11m single (mean of seeds)                                         1 0.3